In [2]:
%pwd

'c:\\Users\\noyon\\OneDrive\\Desktop\\Medical-assistant\\Medical-Chatbot\\research'

In [3]:
import os
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\noyon\\OneDrive\\Desktop\\Medical-assistant\\Medical-Chatbot'

In [5]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [6]:
def load_pdf_files(data):
    loader=DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )

    documents=loader.load()
    return documents

In [7]:
extracted_data=load_pdf_files("data")

In [9]:
len(extracted_data)

637

In [10]:
from typing import List
from langchain.schema import Document

def filter_to_minimal_docs(docs: List[Document])-> List[Document]:
    minimal_docs: List[Document]=[]
    for doc in docs:
        src=doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source":src}
            )
        )
    return minimal_docs

In [11]:
minimal_docs=filter_to_minimal_docs(extracted_data)

In [15]:
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20,
    )
    texts_chunk = text_splitter.split_documents(minimal_docs)
    return texts_chunk

In [16]:
texts_chunk=text_split(minimal_docs)
print(f"Number of chunks: {len(texts_chunk)}")

Number of chunks: 5859


In [17]:
from langchain.embeddings import HuggingFaceEmbeddings

def download_embeddings():
    model_name="sentence-transformers/all-MiniLM-L6-v2"
    embeddings=HuggingFaceEmbeddings(
        model_name=model_name
    )
    return embeddings
embedding=download_embeddings()


C:\Users\noyon\AppData\Local\Temp\ipykernel_11928\2296253828.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings=HuggingFaceEmbeddings(


In [18]:
vector=embedding.embed_query("Hello world")
vector

[-0.03447720408439636,
 0.031023239716887474,
 0.00673496862873435,
 0.026108969002962112,
 -0.03936196118593216,
 -0.16030246019363403,
 0.06692393124103546,
 -0.0064414795488119125,
 -0.047450557351112366,
 0.014758911915123463,
 0.0708753690123558,
 0.05552756413817406,
 0.01919337548315525,
 -0.026251327246427536,
 -0.010109500028192997,
 -0.026940541341900826,
 0.022307470440864563,
 -0.02222665585577488,
 -0.14969269931316376,
 -0.01749308407306671,
 0.007676247972995043,
 0.054352279752492905,
 0.003254473675042391,
 0.03172597661614418,
 -0.0846213549375534,
 -0.0294059906154871,
 0.051595624536275864,
 0.048124030232429504,
 -0.003314792178571224,
 -0.05827920511364937,
 0.04196930304169655,
 0.022210685536265373,
 0.1281888484954834,
 -0.02233896590769291,
 -0.011656301096081734,
 0.06292833387851715,
 -0.032876282930374146,
 -0.09122605621814728,
 -0.031175389885902405,
 0.05269956216216087,
 0.0470348559319973,
 -0.08420302718877792,
 -0.03005620837211609,
 -0.0207447819411

In [19]:
print(len(vector))

384


In [20]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [21]:
PINECONE_API_KEY=os.getenv("PINECONE_API_KEY")
OPENROUTER_API_KEY=os.getenv("OPENROUTER_API_KEY")
os.environ["PINECONE_API_KEY"]=PINECONE_API_KEY
os.environ["OPENROUTER_API_KEY"]=OPENROUTER_API_KEY

In [22]:
from pinecone import Pinecone
pinecone_api_key=PINECONE_API_KEY
pc=Pinecone(api_key=pinecone_api_key)

In [24]:
from pinecone import ServerlessSpec
index_name="medical-chatbot"

if index_name not in pc.list_indexes().names():
        pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws",region="us-east-1")
    )
index=pc.Index(index_name)

In [25]:
from langchain_pinecone import PineconeVectorStore

docsearch=PineconeVectorStore.from_documents(
    documents=texts_chunk,
    embedding=embedding,
    index_name=index_name
)

In [26]:
from langchain_pinecone import PineconeVectorStore
docsearch=PineconeVectorStore.from_existing_index(
    embedding=embedding,
    index_name=index_name
)

In [27]:
retriever=docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [28]:
retrieved_docs=retriever.invoke("What is acne?")
retrieved_docs

[Document(id='d2710808-4f49-4d82-ac2f-a371e1eae006', metadata={'source': 'data\\Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='d1cc3b4c-6996-4497-b0d0-d653f872ec5f', metadata={'source': 'data\\Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 2 25\nAcne\nAcne vulgaris affecting a woman’s face. Acne is the general\nname given to a skin disorder in which the sebaceous\nglands become inflamed. (Photograph by Biophoto Associ-\nates, Photo Researchers, Inc. Reproduced by permission.)\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 25'),
 Document(id='32a51757-1267-4c7c-b45f-fdfefd9a4c7b', metadata={'source': 'data\\Medical_book.pdf'}, page_content='Acidosis see Respiratory acidosis; Renal\ntubular acidosis; Metabolic acidosis\nAcne\nDefinition\nAcne is a common skin disease characterized by\npimples on the face, chest, and back. It occurs when the\npores of the skin become clogged wi

In [30]:
from langchain_openai import ChatOpenAI
chatModel=ChatOpenAI(base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
    model="openai/gpt-oss-120b:free")

In [31]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [32]:
system_prompt=(
    "You are a Medical assistant for question-answering tasks. "
    "Use teh following pieces of retrieved context to answer the question. "
    "If you don't know the answer, say thank you don't know. Use three sentences maximum and keep the answer concise."
    "\n\n"
    "{context}"
)

In [33]:
prompt=ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human","{input}"),
    ]
)

In [34]:
question_answer_chain=create_stuff_documents_chain(chatModel, prompt)
rag_chain=create_retrieval_chain(retriever, question_answer_chain)

In [37]:
response=rag_chain.invoke({"input": "i am feeling prickly sensation on my back what to do?"})
print(response["answer"])

If you notice a new prickly, burning or stinging sensation on your back—especially if it’s accompanied by rash, swelling, or worsening pain—you should contact your physician right away for evaluation. This could signal irritation or infection that needs prompt treatment. Do not delay seeking medical advice.
